# **HANDS - ON: FUNDAMENTOS DE LLMS CON MODELOS LLAMA**


Una vez vista la masterclass ***Fundamentos de LLMs y Arquitectura de Llama***, se proporciona el siguiente ***Colab*** para construir, en vivo, la primera llamada a Llama y medir su comportamiento.

Usamos **Ollama** para ejecutar Llama directamente en la GPU gratuita de Colab, sin necesidad de API key, cuentas externas ni infraestructura propia.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/1bjMHw7fmMxo9dyQp140e9mqotKRkfKBg?usp=sharing).

## **CONFIGURACIÓN DEL ENTORNO**

Antes de llamar a Llama necesitamos activar la GPU gratuita de Colab, instalar **Ollama** (el programa que descarga y ejecuta los modelos) y descargar los modelos que vamos a usar. La librería `ollama` es el cliente oficial en Python para hablar con ese servidor.

### **ACTIVAR LA GPU**

En el menú de Colab ve a **Entorno de ejecución → Cambiar tipo de entorno de ejecución** y selecciona **T4 GPU**. Luego ejecuta la celda para confirmar que Colab te asignó la GPU.

In [ ]:
# Verificar que Colab asignó una GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv




name, memory.total [MiB]
Tesla T4, 15360 MiB


### **INSTALAR OLLAMA**

A diferencia de una API en la nube, aquí **el modelo corre dentro de tu propia sesión de Colab**: no hay *API key* ni *Secrets*. Ollama se instala en la máquina virtual, se arranca como un servidor local (`localhost:11434`) y desde Python le enviamos los prompts.

**IMPORTANTE:** la sesión de Colab es temporal. Si se reinicia o se desconecta, hay que volver a ejecutar las celdas de instalación, servidor y descarga de modelos.

In [ ]:
# Instalar Ollama en la máquina virtual de Colab (tarda ~1 minuto)
!sudo apt-get -qq install -y zstd > /dev/null 2>&1

!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

print("Ollama instalado.")


Ollama instalado.


In [ ]:
# Arrancar el servidor de Ollama en segundo plano y esperar a que responda
import subprocess

import time

import requests

servidor = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(30):

    try:

        requests.get("http://localhost:11434")

        print("Servidor de Ollama listo.")

        break

    except requests.exceptions.ConnectionError:

        time.sleep(1)

else:

    print("El servidor no respondió. Vuelve a ejecutar esta celda.")

!ollama --version


Servidor de Ollama listo.
ollama version is 0.40.3


### **DESCARGAR LOS MODELOS**

Vamos a trabajar con dos tamaños de Llama: **Llama 3.2 de 3B** (ligero) y **Llama 3.1 de 8B** (más grande). Guardamos los nombres en variables para usar *exactamente la misma etiqueta* al descargar y al llamar al modelo; si no coinciden carácter por carácter, Ollama devuelve error.

In [ ]:
# Descargar los dos modelos Llama (la primera vez tarda unos minutos)
MODELO_LIGERO = "llama3.2:3b"

MODELO_GRANDE = "llama3.1:8b"

!ollama pull {MODELO_LIGERO}

!ollama pull {MODELO_GRANDE}

!ollama list



NAME           ID              SIZE      MODIFIED               
llama3.1:8b    46e0c10c039e    4.9 GB    Less than a second ago    
llama3.2:3b    a80c4f17acd5    2.0 GB    52 seconds ago            


### **CLIENTE DE PYTHON**

In [ ]:
# Instalar el cliente de Ollama y conectarlo al servidor local
!pip install ollama -q

import ollama

import time

client = ollama.Client(host="http://localhost:11434")

print("Cliente de Ollama inicializado correctamente.")

Cliente de Ollama inicializado correctamente.


## **DE PALABRAS A TOKENS**

### **PRIMERA LLAMADA A LLAMA (ZERO-SHOT)**

Un LLM predice el siguiente token más probable, **no busca la respuesta en una base de datos**. Antes de que Llama genere una sola palabra, tu prompt se divide en **tokens**.

Vamos a enviar el prompt a Llama en modo zero-shot (sin ejemplos previos) y ver la respuesta generada.

In [ ]:
# Definir y enviar un prompt a Llama en Ollama y mostrar la respuesta generada
prompt = "¿Cuál es la diferencia entre la RAM y el almacenamiento en una computadora?"

response = client.chat(

    model=MODELO_LIGERO,

    messages=[{"role": "user", "content": prompt}]

)

print(prompt)

print(response.message.content)

¿Cuál es la diferencia entre la RAM y el almacenamiento en una computadora?
**Diferencia entre RAM y almacenamiento en una computadora**

La RAM (Random Access Memory) y el almacenamiento son dos componentes fundamentales de una computadora que desempeñan un papel crucial en su funcionamiento.

**RAM (Random Access Memory)**

La RAM es un tipo de memoria de acceso aleatorio que se utiliza para almacenar datos y programas temporales en la computadora. Es un tipo de memoria volatile, lo que significa que los datos se pierden cuando la computadora se apaga. La RAM es diseñada para realizar tareas que requieren acceso rápido y aleatorio a los datos, como procesar instrucciones y ejecutar programas.

**Almacenamiento**

El almacenamiento, por otro lado, se refiere a cualquier dispositivo o tecnología que almacena datos permanentemente en la computadora. Puede incluir:

* Disco duro (HD) o disco sólido (SSD)
* Tarjeta de memoria externa (SD o CF)
* Dispositivos de almacenamiento en la nube (

In [ ]:
print(response.model_dump_json(indent=2))

{
  "model": "llama3.2:3b",
  "created_at": "2026-10-14T01:45:56.482168347Z",
  "done": true,
  "done_reason": "stop",
  "total_duration": 5925830680,
  "load_duration": 4776569,
  "prompt_eval_count": 43,
  "prompt_eval_duration": 34998000,
  "eval_count": 476,
  "eval_duration": 5835449000,
  "message": {
    "role": "assistant",
    "content": "**Diferencia entre RAM y almacenamiento en una computadora**\n\nLa RAM (Random Access Memory) y el almacenamiento son dos componentes fundamentales de una computadora que desempeñan un papel crucial en su funcionamiento.\n\n**RAM (Random Access Memory)**\n\nLa RAM es un tipo de memoria de acceso aleatorio que se utiliza para almacenar datos y programas temporales en la computadora. Es un tipo de memoria volatile, lo que significa que los datos se pierden cuando la computadora se apaga. La RAM es diseñada para realizar tareas que requieren acceso rápido y aleatorio a los datos, como procesar instrucciones y ejecutar programas.\n\n**Almacenamie

### **CUÁNTOS TOKENS CONSUMIÓ EL PROMPT**

La respuesta de Ollama trae el conteo de tokens de entrada (`prompt_eval_count`) y de salida (`eval_count`). Es la forma más directa de responder: ¿cuántos tokens consumió mi prompt?

In [ ]:
# Mostrar cuántos tokens tuvo el prompt y cuántos tuvo la respuesta
tokens_prompt = response.prompt_eval_count

tokens_respuesta = response.eval_count

print("Tokens del prompt:", tokens_prompt)

print("Tokens de la respuesta:", tokens_respuesta)

print("Tokens totales:", tokens_prompt + tokens_respuesta)

# Con Ollama no se factura: los tokens miden cuánto trabajo hizo el modelo y cuánto de su contexto usaste

Tokens del prompt: 43
Tokens de la respuesta: 476
Tokens totales: 519


### **MIDIENDO LA LATENCIA DE INFERENCIA**

Elegir el tamaño correcto de modelo es una decisión de ingeniería, no solo de potencia bruta. Vamos a medir cuánto tarda Llama en responder el mismo prompt.

In [ ]:
# Medir el tiempo de respuesta de Llama para el mismo prompt
inicio = time.time()

response_tiempo = client.chat(

    model=MODELO_LIGERO,

    messages=[{"role": "user", "content": prompt}]

)

duracion = time.time() - inicio

# duracion_ms = duracion * 1000  # si prefieres reportarlo en milisegundos

print(f"Tiempo de respuesta: {duracion:.2f} segundos")

Tiempo de respuesta: 9.28 segundos


### **COMPARANDO DOS TAMAÑOS DE MODELO**

Repite la misma llamada usando la versión más grande de Llama que cabe en la GPU gratuita de Colab (8B) y compara tiempo de respuesta y calidad contra el modelo ligero (3B).

Ojo: la primera vez que se usa un modelo, Ollama tiene que **cargarlo en la memoria de la GPU**. Ese tiempo de carga viene en `load_duration` y conviene separarlo para que la comparación sea justa.

In [ ]:
# Definir y enviar un prompt a Llama en Ollama y mostrar la respuesta generada
inicio = time.time()

response_grande = client.chat(

    model=MODELO_GRANDE,

    messages=[{"role": "user", "content": prompt}]

)

duracion_grande = time.time() - inicio

carga_grande = response_grande.load_duration / 1e9  # nanosegundos -> segundos

tokens_ligero = response.prompt_eval_count + response.eval_count

tokens_grande = response_grande.prompt_eval_count + response_grande.eval_count

print(f"Modelo ligero ({MODELO_LIGERO}): {duracion:.2f} s — {tokens_ligero} tokens")

print(f"Modelo grande ({MODELO_GRANDE}): {duracion_grande:.2f} s — {tokens_grande} tokens (de los cuales {carga_grande:.2f} s fueron carga del modelo)")

print("\nRespuesta del modelo grande:\n", response_grande.message.content)

Modelo ligero (llama3.2:3b): 9.28 s — 519 tokens
Modelo grande (llama3.1:8b): 22.59 s — 190 tokens (de los cuales 18.26 s fueron carga del modelo)

Respuesta del modelo grande:
 La RAM (memoria de acceso aleatorio) y el almacenamiento en una computadora son componentes separados que cumplen propósitos diferentes.

La RAM es una tecnología de memoria temporal que se utiliza para almacenar datos del sistema operativo y aplicaciones ejecutándose actualmente. Es una memoria volátil, lo que significa que los datos se eliminan una vez que la computadora se apaga.

Por otro lado, el almacenamiento es un tipo de memoria no volátil que almacena datos y programas de manera permanente, incluso cuando la computadora se apaga. Almacenamiento puede ser en forma de disco duro, disco duro sólido (SSD), unidad de estado sólido (SSD) u otros tipos de almacenamiento.


## **CHALLENGE: COMPARADOR DE MODELOS LLAMA**

Una vez visto el ***Hands-On: Fundamentos de LLMs***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se construirá un pequeño **comparador que envíe 3 preguntas** reales a Llama y registre, para cada una, el **tiempo de respuesta** y el **número de tokens** consumidos.

**IMPORTANTE:** Para su revisión, **es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.**

### **INSTRUCCIONES:**

**1. Carga la API key y genera 3 preguntas:** Prepara el entorno de configuración del entorno con **Ollama** activando la **T4 GPU** e inicializa el cliente.

In [1]:
# Instalar Ollama en la máquina virtual de Colab (tarda ~1 minuto)
!sudo apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

print("Ollama instalado.")

Ollama instalado.


In [2]:
# Arrancar el servidor de Ollama en segundo plano y esperar a que responda
import subprocess
import time
import requests

servidor = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(30):
    try:
        requests.get("http://localhost:11434")
        print("Servidor de Ollama listo.")
        break
    except requests.exceptions.ConnectionError:
        time.sleep(1)
else:
    print("El servidor no respondió. Vuelve a ejecutar esta celda.")

!ollama --version

Servidor de Ollama listo.
ollama version is 0.40.6


In [9]:
# Descargar el modelo ligero de Llama e inicializar el cliente
MODELO_LIGERO = "llama3.2:3b"
!ollama pull {MODELO_LIGERO}
!ollama list
!pip install -q ollama
import ollama
client = ollama.Client(host="http://localhost:11434")
print("Cliente inicializado.")


NAME           ID              SIZE      MODIFIED               
llama3.2:3b    a80c4f17acd5    2.0 GB    Less than a second ago    
Cliente inicializado.


**2. Consulta la primera pregunta:**

* Construye una lista llamada `preguntas` y agrégale 3 preguntas frecuentes de tu propio contexto (soporte técnico, tienda, escuela, etc.).
*  Envía la primera pregunta a Llama usando el modelo más ligero disponible en Groq. Guarda la respuesta, el tiempo de respuesta y el número de tokens en un diccionario llamado `resultado_1`.

In [12]:
# Definir la lista de preguntas
preguntas = [
    "¿Cuál es la diferencia entre RAM y ROM?",
    "¿Cuál es la diferencia entre CPU y GPU?",
    "¿Cuál es la diferencia entre disco duro mecánico y SSD?"
]
for i, p in enumerate(preguntas, 1):
    print(f"{i}. {p}")

1. ¿Cuál es la diferencia entre RAM y ROM?
2. ¿Cuál es la diferencia entre CPU y GPU?
3. ¿Cuál es la diferencia entre disco duro mecánico y SSD?


In [13]:
# Consultar la pregunta 1 y guardar el resultado en resultado_1
import time

inicio = time.time()
response = client.chat(
    model=MODELO_LIGERO,
    messages=[{"role": "user", "content": preguntas[0]}]
)
fin = time.time()

resultado_1 = {
    "pregunta": preguntas[0],
    "respuesta": response.message.content,
    "tiempo_segundos": round(fin - inicio, 2),
    "tokens": response.eval_count  # tokens generados en la respuesta
}

print(resultado_1)

{'pregunta': '¿Cuál es la diferencia entre RAM y ROM?', 'respuesta': '¡Hola! Me alegra poder ayudarte con esa pregunta.\n\nRAM (Random Access Memory) y ROM (Read-Only Memory) son dos tipos de memoria de computadora que tienen funciones y características muy diferentes.\n\n**RAM (Random Access Memory)**\n\nLa RAM es una tipo de memoria que almacena los datos y los programas actuales que se están utilizando en el computador. La RAM es voluble, lo que significa que los datos almacenados en ella pueden ser modificados y eliminados.\n\nLas características clave de la RAM son:\n\n*   Es voluble.\n*   Los datos almacenados en ella pueden ser modificados.\n*   Los datos se cargan y se descargan rápidamente.\n*   Es temporal, lo que significa que se vuelve a inicializar cada vez que el computador se apaga.\n\n**ROM (Read-Only Memory)**\n\nLa ROM es una tipo de memoria que almacena datos permanentes y no modificables. La ROM es inmutable, lo que significa que los datos almacenados en ella no pue

**3. Repite para la segunda y tercera pregunta:** Crea `resultado_2` y `resultado_3` de la misma forma.

In [14]:
# Consultar la pregunta 2 y guardar el resultado en resultado_2
inicio = time.time()
response = client.chat(
    model=MODELO_LIGERO,
    messages=[{"role": "user", "content": preguntas[1]}]
)
fin = time.time()

resultado_2 = {
    "pregunta": preguntas[1],
    "respuesta": response.message.content,
    "tiempo_segundos": round(fin - inicio, 2),
    "tokens": response.eval_count
}

print(resultado_2)

{'pregunta': '¿Cuál es la diferencia entre CPU y GPU?', 'respuesta': '**Diferencia entre CPU y GPU**\n\nEn el mundo de la tecnología, dos componentes son fundamentales para el rendimiento de los ordenadores: el procesador central unitario (CPU) y la unidad gráfica (GPU). Aunque ambos componentes son esenciales, tienen funciones y características muy diferentes.\n\n**CPU (Procesador Central Unitario)**\n\nLa CPU es el " cerebro" del ordenador, responsable de ejecutar instrucciones y realizar cálculos. Su función principal es:\n\n* Executar instrucciones y realizar cálculos\n* Gestionar el flujo de datos entre componentes\n* Controlar el sistema operativo y las aplicaciones\n\nLas CPU son diseñadas para realizar tareas de procesamiento de datos y ejecución de instrucciones, y están optimizadas para realizar tareas como:\n\n* Ejecución de código de software\n* Procesamiento de datos\n* Gestión de sistemas operativos\n\n**GPU (Unidad Gráfica)**\n\nLa GPU, por otro lado, es especializada en

In [15]:
# Consultar la pregunta 3 y guardar el resultado en resultado_3
inicio = time.time()
response = client.chat(
    model=MODELO_LIGERO,
    messages=[{"role": "user", "content": preguntas[2]}]
)
fin = time.time()

resultado_3 = {
    "pregunta": preguntas[2],
    "respuesta": response.message.content,
    "tiempo_segundos": round(fin - inicio, 2),
    "tokens": response.eval_count
}

print(resultado_3)

{'pregunta': '¿Cuál es la diferencia entre disco duro mecánico y SSD?', 'respuesta': 'La principal diferencia entre un disco duro mecánico y un SSD (Solid State Drive) es la forma en que se almacenan y acceden a los datos.\n\n**Disco Duro Mecánico**\n\nUn disco duro mecánico utiliza discos rotatorios que giran a alta velocidad para almacenar datos en forma de cintas magnéticas. Los discos rotatorios están conectados a una cabeza de lectura/escritura (HLD) que se mueve sobre el disco para acceder a los datos.\n\n**Ventajas del disco duro mecánico**\n\n* Almacenamiento de grandes cantidades de datos\n* Costo relativamente bajo\n* Velocidad de transferencia de datos moderada\n\n**Desventajas del disco duro mecánico**\n\n* Inestabilidad del sistema de refrigeración\n* Ruido y vibración\n* Tiempo de respuesta lento\n* Consumo de energía alto\n* Conocimiento de la ubicación de los datos es más difícil\n\n**SSD (Solid State Drive)**\n\nUn SSD utiliza memoria flash para almacenar datos en form

**4. Junta los resultados:** Agrega `resultado_1`, `resultado_2` y `resultado_3` a una lista vacía llamada `resultados`.

In [16]:
# Definir la lista resultados y agregar los tres diccionarios
resultados = []

resultados.append(resultado_1)
resultados.append(resultado_2)
resultados.append(resultado_3)

print(f"Resultados guardados: {len(resultados)}")

Resultados guardados: 3


**5. Muestra la comparación:** Imprime `resultados` y concluye si el modelo ligero resolvió las 3 preguntas satisfactoriamente.

In [17]:
# Mostrar la tabla final de resultados
import pandas as pd

print(resultados)  # imprime la lista, como pide la instrucción

tabla = pd.DataFrame(resultados)
tabla

[{'pregunta': '¿Cuál es la diferencia entre RAM y ROM?', 'respuesta': '¡Hola! Me alegra poder ayudarte con esa pregunta.\n\nRAM (Random Access Memory) y ROM (Read-Only Memory) son dos tipos de memoria de computadora que tienen funciones y características muy diferentes.\n\n**RAM (Random Access Memory)**\n\nLa RAM es una tipo de memoria que almacena los datos y los programas actuales que se están utilizando en el computador. La RAM es voluble, lo que significa que los datos almacenados en ella pueden ser modificados y eliminados.\n\nLas características clave de la RAM son:\n\n*   Es voluble.\n*   Los datos almacenados en ella pueden ser modificados.\n*   Los datos se cargan y se descargan rápidamente.\n*   Es temporal, lo que significa que se vuelve a inicializar cada vez que el computador se apaga.\n\n**ROM (Read-Only Memory)**\n\nLa ROM es una tipo de memoria que almacena datos permanentes y no modificables. La ROM es inmutable, lo que significa que los datos almacenados en ella no pu

,pregunta,respuesta,tiempo_segundos,tokens
0,¿Cuál es la diferencia entre RAM y ROM?,¡Hola! Me alegra poder ayudarte con esa pregun...,5.47,421
1,¿Cuál es la diferencia entre CPU y GPU?,**Diferencia entre CPU y GPU**\n\nEn el mundo ...,8.39,641
2,¿Cuál es la diferencia entre disco duro mecáni...,La principal diferencia entre un disco duro me...,6.01,455
